In [1]:
import pandas as pd
deirectoryPath ="../data/"

In [2]:
dataDF = pd.read_csv(f"{deirectoryPath}NF-UNSW-NB15-v2.csv")

In [3]:
dataDF.head()

,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,...,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
0,59.166.0.5,1305,149.171.126.8,21,6,1.0,9,1,193,3,...,0,7240,0,0,0,0,0,331.0,0,Benign
1,59.166.0.5,1305,149.171.126.8,21,6,1.0,261,5,469,7,...,8688,8688,18944,74,0,0,0,230.0,0,Benign
2,59.166.0.5,1305,149.171.126.8,21,6,1.0,481,9,750,11,...,10136,10136,33792,132,0,0,0,229.0,0,Benign
3,59.166.0.5,1305,149.171.126.8,21,6,1.0,701,13,1054,15,...,11584,11584,48640,190,0,0,0,125.0,0,Benign
4,59.166.0.5,1305,149.171.126.8,21,6,1.0,1031,19,1474,21,...,14480,13032,64256,251,0,0,0,230.0,0,Benign


In [4]:
len(dataDF.columns)

45

In [5]:
len(dataDF['IPV4_SRC_ADDR'].unique())

40

In [6]:
len(dataDF['IPV4_DST_ADDR'].unique())

40

In [8]:
dataDF.drop(columns=["Attack" , "L4_SRC_PORT"] , inplace = True , axis = 1)
dataDF.head()

,IPV4_SRC_ADDR,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,...,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label
0,59.166.0.5,149.171.126.8,21,6,1.0,9,1,193,3,24,...,0,0,7240,0,0,0,0,0,331.0,0
1,59.166.0.5,149.171.126.8,21,6,1.0,261,5,469,7,24,...,0,8688,8688,18944,74,0,0,0,230.0,0
2,59.166.0.5,149.171.126.8,21,6,1.0,481,9,750,11,24,...,0,10136,10136,33792,132,0,0,0,229.0,0
3,59.166.0.5,149.171.126.8,21,6,1.0,701,13,1054,15,24,...,0,11584,11584,48640,190,0,0,0,125.0,0
4,59.166.0.5,149.171.126.8,21,6,1.0,1031,19,1474,21,24,...,0,14480,13032,64256,251,0,0,0,230.0,0


In [9]:
from sklearn.model_selection import train_test_split

trainDF , testDF = train_test_split(dataDF , test_size=0.25 , random_state=20 , stratify=dataDF['Label'])

In [10]:
import networkx as netx
trainingGraph = netx.MultiDiGraph()
testingGraph = netx.MultiDiGraph()

In [11]:
dataDF.columns

Index(['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO',
       'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS',
       'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS',
       'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT',
       'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
       'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES',
       'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS',
       'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS',
       'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
       'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
       'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
       'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
       'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE',
       'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Label'],
      dtype='object')

In [12]:
srcTrain = trainDF['IPV4_SRC_ADDR'].to_list()
destTrain = trainDF['IPV4_DST_ADDR'].to_list() 
srcTest = testDF['IPV4_SRC_ADDR'].to_list()
destTest = testDF['IPV4_DST_ADDR'].to_list() 

In [13]:
trainingGraph.add_nodes_from(srcTrain)
trainingGraph.add_nodes_from(destTrain)
testingGraph.add_nodes_from(srcTest)
testingGraph.add_nodes_from(destTest)

In [14]:
feature_cols = [col for col in trainDF.columns if col not in ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label']]

In [15]:
from sklearn.preprocessing import StandardScaler
scalerEdge = StandardScaler()
train_feats = scalerEdge.fit_transform(trainDF[feature_cols].values.astype(float))
test_feats  = scalerEdge.transform(testDF[feature_cols].values.astype(float)) 

In [16]:
for i, (idx, row) in enumerate(trainDF.iterrows()):
    trainingGraph.add_edge(row['IPV4_SRC_ADDR'], row['IPV4_DST_ADDR'],
                            features=train_feats[i], label=row['Label'])

In [17]:
for i, (idx, row) in enumerate(testDF.iterrows()):
    testingGraph.add_edge(row['IPV4_SRC_ADDR'], row['IPV4_DST_ADDR'],
                            features=test_feats[i], label=row['Label'])

In [18]:
print(f"Training Graph - Nodes: {trainingGraph.number_of_nodes()}, Edges: {trainingGraph.number_of_edges()}")
print(f"Test Graph - Nodes: {testingGraph.number_of_nodes()}, Edges: {testingGraph.number_of_edges()}")
print(f"Edge features shape: {trainingGraph[list(trainingGraph.edges())[0][0]][list(trainingGraph.edges())[0][1]][0]['features'].shape}")

Training Graph - Nodes: 43, Edges: 1792706
Test Graph - Nodes: 42, Edges: 597569
Edge features shape: (40,)


In [19]:
src_stats_train = trainDF.groupby("IPV4_SRC_ADDR").agg(
    out_degree=("IPV4_SRC_ADDR", "count"),
    avg_sbytes=("IN_BYTES", "mean"),
    avg_spkts=("IN_PKTS", "mean")
)

dst_stats_train = trainDF.groupby("IPV4_DST_ADDR").agg(
    in_degree=("IPV4_DST_ADDR", "count"),
    avg_dbytes=("OUT_BYTES", "mean"),
    avg_dpkts=("OUT_PKTS", "mean")
)

In [20]:
src_stats_test = testDF.groupby("IPV4_SRC_ADDR").agg(
    out_degree=("IPV4_SRC_ADDR", "count"),
    avg_sbytes=("IN_BYTES", "mean"),
    avg_spkts=("IN_PKTS", "mean")
)

dst_stats_test = testDF.groupby("IPV4_DST_ADDR").agg(
    in_degree=("IPV4_DST_ADDR", "count"),
    avg_dbytes=("OUT_BYTES", "mean"),
    avg_dpkts=("OUT_PKTS", "mean")
)

In [21]:
trainNodeFeature = pd.concat(
    [src_stats_train, dst_stats_train],
    axis=1
)

trainNodeFeature = trainNodeFeature.fillna(0)

In [ ]:
testNodeFeature = pd.concat(
    [src_stats_test, dst_stats_test],
    axis=1
)

testNodeFeature = trainNodeFeature.fillna(0)

In [23]:
nodeScaler = StandardScaler()
trainNodeFeature[:] = nodeScaler.fit_transform(trainNodeFeature)

In [24]:
testNodeFeature[:] = nodeScaler.transform(testNodeFeature)

In [25]:
import torch
from torch_geometric.utils import from_networkx

trainGraphData = from_networkx(trainingGraph)
testGraphData = from_networkx(testingGraph)

/mnt/d/Personal_project/DeepLearning/nnForGraphs/attackClassifier/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/d/Personal_project/DeepLearning/nnForGraphs/attackClassifier/venv/lib/python3.10/site-packages/torch_geometric/utils/convert.py:278: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  data_dict[key] = torch.as_tensor(value)


In [26]:
trainGraphData.nodeFeature = trainNodeFeature
testGraphData.nodeFeature = testNodeFeature

In [27]:
edge_features_train = []
edge_labels_train = []

for _ , _ , attr in trainingGraph.edges(data=True):
    edge_features_train.append(attr['features'])
    edge_labels_train.append(attr['label'])

In [28]:
edge_features_test = []
edge_labels_test = []

for _ , _ , attr in testingGraph.edges(data=True):
    edge_features_train.append(attr['features'])
    edge_labels_train.append(attr['label'])

In [29]:
len(edge_features_train[0])

40

In [30]:
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [31]:
trainGraphData.edge_attr = torch.tensor(
    edge_features_train,
    dtype=torch.float
).to(device)

trainGraphData.edge_label = torch.tensor(
    edge_labels_train,
    dtype=torch.float
).to(device)

In [32]:
testGraphData.edge_attr = torch.tensor(
    edge_features_test,
    dtype=torch.float
).to(device)

testGraphData.edge_label = torch.tensor(
    edge_labels_test,
    dtype=torch.float
).to(device)